-SCT213-C002-0141/2024 GACHOMO NJERU

-SCT213-C002-0063/2024 MWITA NYEHITA

-SCT213-C002-0142/2024 KARWEGA NJERU
# Sprint 2:  Milestone 2 : Data Processing & Transformation


**Objectives:**
- Data cleaning (missing values, anomalies)
- Transformation (aggregation, grouping, joins)
- Feature engineering


## 1. Load Dataset

In [ ]:
#SCT213-C002-0141/2024 GACHOMO NJERU
#SCT213-C002-0063/2024 MWITA NYEHITA
#SCT213-C002-0142/2024 KARWEGA NJERU
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the tips dataset (same as Sprint 1)
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv'
df = pd.read_csv(url)

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
df.head()

## 2. Data Cleaning
### . Check for Missing Values

In [ ]:
print("=== Missing Values ===")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

# Artificially inject some missing values to demonstrate cleaning
df_dirty = df.copy()
np.random.seed(42)
missing_indices = np.random.choice(df_dirty.index, size=10, replace=False)
df_dirty.loc[missing_indices[:5], 'tip'] = np.nan
df_dirty.loc[missing_indices[5:], 'total_bill'] = np.nan

print("\n=== After Injecting Missing Values ===")
print(df_dirty.isnull().sum())

### . Handle Missing Values

In [ ]:
df_clean = df_dirty.copy()

# Fill missing 'tip' with the median tip value
median_tip = df_clean['tip'].median()
df_clean['tip'].fillna(median_tip, inplace=True)

# Fill missing 'total_bill' with the mean total_bill value
mean_bill = df_clean['total_bill'].mean()
df_clean['total_bill'].fillna(mean_bill, inplace=True)

print(f"Filled missing 'tip' with median: {median_tip:.2f}")
print(f"Filled missing 'total_bill' with mean: {mean_bill:.2f}")
print("\n=== Missing Values After Cleaning ===")
print(df_clean.isnull().sum())

### . Detect and Handleing Anomalies 

In [ ]:
# Detect outliers using IQR method
def detect_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = series[(series < lower) | (series > upper)]
    return outliers, lower, upper

for col in ['total_bill', 'tip']:
    outliers, lower, upper = detect_outliers_iqr(df_clean[col])
    print(f"\n{col}: {len(outliers)} outliers found")
    print(f"  Acceptable range: [{lower:.2f}, {upper:.2f}]")
    if len(outliers) > 0:
        print(f"  Outlier values: {outliers.values}")

# Cap outliers (winsorization) instead of removing rows
for col in ['total_bill', 'tip']:
    _, lower, upper = detect_outliers_iqr(df_clean[col])
    df_clean[col] = df_clean[col].clip(lower=lower, upper=upper)

print(" Outliers capped using IQR winsorization.")

### . Remove Duplicate Rows

In [ ]:
print(f"Duplicate rows before: {df_clean.duplicated().sum()}")
df_clean = df_clean.drop_duplicates()
print(f"Duplicate rows after:  {df_clean.duplicated().sum()}")
print(f"\nClean dataset shape: {df_clean.shape}")

## 3. Transformations
### 3a. Aggregation — Summary Statistics by Group

In [ ]:
# aggregation by day
agg_by_day = df_clean.groupby('day').agg(
    avg_tip=('tip', 'mean'),
    avg_bill=('total_bill', 'mean'),
    total_tips=('tip', 'sum'),
    count=('tip', 'count')
).round(2).reset_index()

print("=== Aggregation by Day ===")
print(agg_by_day.to_string(index=False))

### 3b. Grouping — Multi-level Grouping

In [ ]:
# grouping by time and smoker status
grouped = df_clean.groupby(['time', 'smoker'])['tip'].agg(['mean', 'count']).round(2)
grouped.columns = ['avg_tip', 'count']

print("=== Tips by Time and Smoker Status ===")
print(grouped)

### 3c. Joining — Merge Aggregated Data Back to Original

In [ ]:
# creating a lookup table: average tip per day
day_avg = df_clean.groupby('day')['tip'].mean().round(2).reset_index()
day_avg.columns = ['day', 'day_avg_tip']

# Merge (join) this back onto the main dataframe
df_merged = pd.merge(df_clean, day_avg, on='day', how='left')

print("=== Merged Dataset (sample) ===")
print(df_merged[['total_bill', 'tip', 'day', 'day_avg_tip']].head(10))

## 4. Feature Engineering
### 4a. Create New Derived Features

In [ ]:
df_feat = df_merged.copy()

# Feature 1: Tip percentage
df_feat['tip_pct'] = (df_feat['tip'] / df_feat['total_bill'] * 100).round(2)

# Feature 2: Bill per person
df_feat['bill_per_person'] = (df_feat['total_bill'] / df_feat['size']).round(2)

# Feature 3: Tip vs day average (above or below day average)
df_feat['tip_vs_day_avg'] = (df_feat['tip'] - df_feat['day_avg_tip']).round(2)

# Feature 4: Is weekend? (Sat or Sun)
df_feat['is_weekend'] = df_feat['day'].isin(['Sat', 'Sun']).astype(int)

# Feature 5: Tip generosity category
def tip_category(pct):
    if pct < 10:
        return 'Low (<10%)'
    elif pct < 20:
        return 'Average (10-20%)'
    else:
        return 'Generous (>20%)'

df_feat['tip_category'] = df_feat['tip_pct'].apply(tip_category)

print("=== New Features Added ===")
print(df_feat[['total_bill', 'tip', 'tip_pct', 'bill_per_person',
               'tip_vs_day_avg', 'is_weekend', 'tip_category']].head(10))

### 4b. Encode Categorical Variables

In [ ]:
# Label encode binary categorical columns
df_feat['sex_encoded'] = (df_feat['sex'] == 'Male').astype(int)
df_feat['smoker_encoded'] = (df_feat['smoker'] == 'Yes').astype(int)
df_feat['time_encoded'] = (df_feat['time'] == 'Dinner').astype(int)

print("=== Encoded Categorical Features ===")
print(df_feat[['sex', 'sex_encoded', 'smoker', 'smoker_encoded',
               'time', 'time_encoded']].head(8))

## 5. Clean, Structured Dataset 

In [ ]:
print("=== Final Processed Dataset ===")
print(f"Shape: {df_feat.shape}")
print(f"Columns: {list(df_feat.columns)}")
print()
df_feat.head()

In [ ]:
# Save the clean, structured dataset
df_feat.to_csv('tips_processed.csv', index=False)
print(" Processed dataset saved to tips_processed.csv")

## 6. Visualisation of Processed Data

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Tip % distribution
axes[0].hist(df_feat['tip_pct'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Tip %')
axes[0].set_xlabel('Tip %')
axes[0].set_ylabel('Frequency')

# Plot 2: Average tip by day
agg_by_day.plot(kind='bar', x='day', y='avg_tip', ax=axes[1],
                color='coral', legend=False)
axes[1].set_title('Average Tip by Day')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Average Tip ($)')
axes[1].tick_params(axis='x', rotation=0)

# Plot 3: Tip category counts
df_feat['tip_category'].value_counts().plot(kind='bar', ax=axes[2],
                                             color='mediumseagreen', edgecolor='white')
axes[2].set_title('Tip Generosity Categories')
axes[2].set_xlabel('Category')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('sprint2_visuals.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Visualisations saved to sprint2_visuals.png")

# 7. Documentation of Transformations

| Step | Operation | Description |
|------|-----------|-------------|
| 1 | **Missing value handling** | Injected 10 missing values; filled `tip` with median, `total_bill` with mean |
| 2 | **Outlier detection (IQR)** | Identified outliers using Q1/Q3 ± 1.5×IQR; capped via winsorization |
| 3 | **Duplicate removal** | Dropped any exact duplicate rows |
| 4 | **Aggregation** | Grouped by `day` to compute avg tip, avg bill, total tips, count |
| 5 | **Multi-level grouping** | Grouped by `time` × `smoker` for average tip |
| 6 | **Join / Merge** | Left-joined day-average tip lookup back onto main dataset |
| 7 | **Feature: tip_pct** | Tip as % of total bill |
| 8 | **Feature: bill_per_person** | Total bill divided by party size |
| 9 | **Feature: tip_vs_day_avg** | Deviation of tip from that day's average |
| 10 | **Feature: is_weekend** | Binary flag (1 = Saturday/Sunday) |
| 11 | **Feature: tip_category** | Categorical label: Low / Average / Generous |
| 12 | **Encoding** | Binary encoding for `sex`, `smoker`, `time` |

